# Smart City Traffic & Accident Analytics
**One notebook, 10 optimized tables (8 dimensions + 2 facts)**

Steps: Load → Profile → Clean → Build schema → Save CSVs + MySQL script → Predictions

Put `traffic_accident_data.csv` and `road_traffic_sensor_data.csv` in `data/`, open from project folder, **Run All**.

## 1. Load

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

for folder in ["output/powerbi", "output/clean", "sql"]:
    os.makedirs(folder, exist_ok=True)

LOCATIONS = ["Downtown", "Suburbs", "Residential Zone", "Highway", "Industrial Area"]
WEATHER = ["Clear", "Rain", "Fog", "Snow", "Storm"]
ROADS = ["Dry", "Wet", "Icy", "Under Construction"]
VEHICLES = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle"]
SEVERITY = ["Minor", "Moderate", "Severe", "Fatal"]
SEVERITY_WEIGHT = {"Minor": 1, "Moderate": 2, "Severe": 3, "Fatal": 4}
PEAK_HOURS = [7, 8, 9, 16, 17, 18, 19]

acc_raw = pd.read_csv("data/traffic_accident_data.csv", dtype=str, skipinitialspace=True)
sen_raw = pd.read_csv("data/road_traffic_sensor_data.csv", dtype=str, skipinitialspace=True)
for df in (acc_raw, sen_raw):
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.strip()
print(f"Loaded: {len(acc_raw)} accidents, {len(sen_raw)} sensor readings")

## 2. Profile

In [ ]:
print("---- ACCIDENTS ----")
print("Missing:", acc_raw.isna().sum()[lambda s: s > 0].to_string() or "none")
print("Duplicates:", acc_raw.Accident_ID.duplicated().sum())
acc_dates = pd.to_datetime(acc_raw.Date_Time, errors="coerce")
print("Bad dates:", acc_dates.isna().sum(), "| Range:", acc_dates.min(), "->", acc_dates.max())
for col, allowed in [("Location", LOCATIONS), ("Weather_Condition", WEATHER), ("Road_Condition", ROADS),
                     ("Vehicle_Type", VEHICLES), ("Accident_Severity", SEVERITY)]:
    odd = acc_raw[col].dropna()[~acc_raw[col].dropna().str.title().isin(allowed)]
    if len(odd) > 0:
        print(f"{col}: {odd.unique().tolist()}")
for col in ["Number_of_Vehicles", "Casualties"]:
    num = pd.to_numeric(acc_raw[col], errors="coerce")
    print(f"{col}: non-numeric = {(num.isna() & acc_raw[col].notna()).sum()}, range = {num.min()} to {num.max()}")

print("\n---- SENSORS ----")
print("Missing:", sen_raw.isna().sum()[lambda s: s > 0].to_string() or "none")
sen_dates = pd.to_datetime(sen_raw.Date_Time, errors="coerce")
print("Bad dates:", sen_dates.isna().sum(), "| Range:", sen_dates.min(), "->", sen_dates.max())
for col in ["Vehicle_Count", "Average_Speed"]:
    num = pd.to_numeric(sen_raw[col], errors="coerce")
    print(f"{col}: non-numeric = {(num.isna() & sen_raw[col].notna()).sum()}, range = {num.min()} to {num.max()}")

## 3. Clean

In [ ]:
acc = acc_raw.rename(columns={"Accident_ID": "accident_id", "Date_Time": "event_ts", "Location": "location",
    "Weather_Condition": "weather", "Road_Condition": "road_condition", "Vehicle_Type": "vehicle_type",
    "Accident_Severity": "severity", "Number_of_Vehicles": "number_of_vehicles", "Casualties": "casualties", "Traffic_Density": "traffic_density"})
acc["accident_id"] = acc.accident_id.str.lower()
acc["event_ts"] = pd.to_datetime(acc.event_ts, errors="coerce")
acc["number_of_vehicles"] = pd.to_numeric(acc.number_of_vehicles, errors="coerce")
acc["casualties"] = pd.to_numeric(acc.casualties, errors="coerce")
for col, allowed in [("location", LOCATIONS), ("weather", WEATHER), ("road_condition", ROADS),
                     ("vehicle_type", VEHICLES), ("severity", SEVERITY)]:
    acc[col] = acc[col].str.title()
    acc[col] = acc[col].where(acc[col].isin(allowed), "Unknown")

bad = (acc.accident_id.isna() | acc.event_ts.isna() | acc.number_of_vehicles.isna() | (acc.number_of_vehicles < 1)
       | acc.casualties.isna() | (acc.casualties < 0) | acc.accident_id.duplicated())
print(f"Rejected: {bad.sum()} of {len(acc)}")
acc[bad].to_csv("output/clean/accident_rejects.csv", index=False)
acc = acc[~bad].copy()
acc["number_of_vehicles"] = acc.number_of_vehicles.astype(int)
acc["casualties"] = acc.casualties.astype(int)
acc["severity_score"] = acc.severity.map(SEVERITY_WEIGHT).fillna(0) * (1 + acc.casualties)
acc["is_high_severity"] = acc.severity.isin(["Severe", "Fatal"]).astype(int)
acc["date_key"] = acc.event_ts.dt.strftime("%Y%m%d").astype(int)
acc["hour"] = acc.event_ts.dt.hour

max_people = acc.vehicle_type.map({"Bicycle": 2, "Motorcycle": 2, "Car": 6, "Truck": 3, "Bus": 60}).fillna(6) * acc.number_of_vehicles
checks = {"fatal_no_casualty": (acc.severity == "Fatal") & (acc.casualties == 0),
          "casualties_over_capacity": acc.casualties > max_people,
          "weather_road_conflict": (acc.weather.isin(["Rain", "Storm", "Snow"]) & (acc.road_condition == "Dry")) | ((acc.weather == "Clear") & (acc.road_condition == "Icy")),
          "snow_ice_in_summer": acc.event_ts.dt.month.isin([6, 7, 8]) & ((acc.weather == "Snow") | (acc.road_condition == "Icy"))}
acc["dq_note"] = ""
for name, mask in checks.items():
    acc.loc[mask, "dq_note"] += name + "; "
acc["dq_note"] = acc.dq_note.str.strip("; ").replace("", "OK")
acc.to_csv("output/clean/accident_clean.csv", index=False)
print("DQ notes:", acc.dq_note.value_counts().to_dict())

In [ ]:
sen = sen_raw.rename(columns={"Sensor_ID": "reading_id", "Location": "location", "Date_Time": "event_ts",
    "Vehicle_Count": "vehicle_count", "Average_Speed": "average_speed", "Congestion_Level": "reported_congestion"})
sen["reading_id"] = sen.reading_id.str.lower()
sen["event_ts"] = pd.to_datetime(sen.event_ts, errors="coerce")
sen["vehicle_count"] = pd.to_numeric(sen.vehicle_count, errors="coerce")
sen["average_speed"] = pd.to_numeric(sen.average_speed, errors="coerce")
for col, allowed in [("location", LOCATIONS), ("reported_congestion", ["Low", "Moderate", "High"])]:
    sen[col] = sen[col].str.title()
    sen[col] = sen[col].where(sen[col].isin(allowed), "Unknown")

bad = sen.reading_id.isna() | sen.event_ts.isna() | sen.reading_id.duplicated()
print(f"Rejected: {bad.sum()} of {len(sen)}")
sen[bad].to_csv("output/clean/sensor_rejects.csv", index=False)
sen = sen[~bad].copy()

sen.loc[sen.vehicle_count < 0, "vehicle_count"] = np.nan
sen.loc[(sen.average_speed <= 0) | (sen.average_speed > 150), "average_speed"] = np.nan
sen["dq_note"] = np.where(sen.vehicle_count.isna() | sen.average_speed.isna(), "imputed", "OK")
for col in ["vehicle_count", "average_speed"]:
    sen[col] = sen[col].fillna(sen.groupby("location")[col].transform("median")).fillna(sen[col].median())
sen["vehicle_count"] = sen.vehicle_count.round().astype(int)

volume = (sen.vehicle_count / sen.vehicle_count.quantile(0.95)).clip(0, 1)
speed = (sen.average_speed / sen.average_speed.quantile(0.90)).clip(0, 1)
sen["congestion_index"] = (100 * (0.5 * volume + 0.5 * (1 - speed))).round(1)
sen["congestion_level"] = pd.cut(sen.congestion_index, [-1, 40, 60, 101], labels=["Low", "Moderate", "High"], right=False).astype(str)
sen["is_high_congestion"] = (sen.congestion_level == "High").astype(int)
sen["date_key"] = sen.event_ts.dt.strftime("%Y%m%d").astype(int)
sen["hour"] = sen.event_ts.dt.hour
sen.to_csv("output/clean/sensor_clean.csv", index=False)
print("DQ notes:", sen.dq_note.value_counts().to_dict())

## 4. Star schema (8 dimensions + 2 facts)

In [ ]:
days = pd.date_range(min(acc.event_ts.min(), sen.event_ts.min()).normalize(), max(acc.event_ts.max(), sen.event_ts.max()).normalize())
dim_date = pd.DataFrame({"date_key": days.strftime("%Y%m%d").astype(int), "full_date": days.strftime("%Y-%m-%d"),
    "year": days.year, "month": days.month, "month_name": days.strftime("%b"), "month_label": days.strftime("%Y-%m"),
    "day": days.day, "dow_num": days.dayofweek, "dow_name": days.strftime("%a"), "is_weekend": (days.dayofweek >= 5).astype(int)})

dim_time = pd.DataFrame({"hour": range(24)})
h = dim_time.hour
dim_time["time_band"] = np.select([h < 6, h < 12, h < 17, h < 22], ["Night", "Morning", "Afternoon", "Evening"], "Night")
dim_time["peak_period"] = np.select([h.isin([7, 8, 9]), h.isin([16, 17, 18, 19])], ["Morning Peak", "Evening Peak"], "Off-Peak")
dim_time["is_peak_hour"] = h.isin(PEAK_HOURS).astype(int)

def make_dim(key, name, values):
    values = values + ["Unknown"]
    return pd.DataFrame({key: range(1, len(values) + 1), name: values})

dim_location = make_dim("location_key", "location_name", LOCATIONS)
dim_weather = make_dim("weather_key", "weather", WEATHER)
dim_road = make_dim("road_key", "road_condition", ROADS)
dim_vehicle = make_dim("vehicle_key", "vehicle_type", VEHICLES)
dim_severity = make_dim("severity_key", "severity", SEVERITY)
dim_severity["severity_weight"] = dim_severity.severity.map(SEVERITY_WEIGHT).fillna(0).astype(int)
dim_congestion = pd.DataFrame({"congestion_key": [1, 2, 3], "congestion_level": ["Low", "Moderate", "High"]})

fact_accident = acc.copy()
for col, dim, name, key in [("location", dim_location, "location_name", "location_key"), ("weather", dim_weather, "weather", "weather_key"),
                            ("road_condition", dim_road, "road_condition", "road_key"), ("vehicle_type", dim_vehicle, "vehicle_type", "vehicle_key"),
                            ("severity", dim_severity, "severity", "severity_key")]:
    fact_accident[key] = fact_accident[col].map(dict(zip(dim[name], dim[key])))
fact_accident["event_ts"] = fact_accident.event_ts.dt.strftime("%Y-%m-%d %H:%M:%S")
fact_accident = fact_accident[["accident_id", "event_ts", "date_key", "hour", "location_key", "weather_key", "road_key", "vehicle_key",
    "severity_key", "number_of_vehicles", "casualties", "traffic_density", "severity_score", "is_high_severity", "dq_note"]]

fact_traffic = sen.copy()
fact_traffic["location_key"] = fact_traffic.location.map(dict(zip(dim_location.location_name, dim_location.location_key)))
fact_traffic["congestion_key"] = fact_traffic.congestion_level.map(dict(zip(dim_congestion.congestion_level, dim_congestion.congestion_key)))
fact_traffic["event_ts"] = fact_traffic.event_ts.dt.strftime("%Y-%m-%d %H:%M:%S")
fact_traffic = fact_traffic[["reading_id", "event_ts", "date_key", "hour", "location_key", "vehicle_count", "average_speed",
    "reported_congestion", "congestion_index", "congestion_key", "is_high_congestion", "dq_note"]]

tables = {"dim_date": dim_date, "dim_time": dim_time, "dim_location": dim_location, "dim_weather": dim_weather, "dim_road": dim_road,
          "dim_vehicle": dim_vehicle, "dim_severity": dim_severity, "dim_congestion_level": dim_congestion,
          "fact_accident": fact_accident, "fact_traffic": fact_traffic}
for name, df in tables.items():
    print(f"{name:22} {len(df):5} rows")

## 5. Save CSVs & SQL script

In [ ]:
for name, df in tables.items():
    df.to_csv(f"output/powerbi/{name}.csv", index=False, encoding="utf-8-sig")

def sql_val(v):
    if pd.isna(v): return "NULL"
    if isinstance(v, (int, float, np.integer, np.floating)): return str(v)
    return "'" + str(v).replace("\\", "\\\\").replace("'", "''") + "'"

with open("sql/02_insert_data.sql", "w", encoding="utf-8") as f:
    f.write("USE smart_city_dw;\nSET FOREIGN_KEY_CHECKS = 0;\n\n")
    for name in ["dim_date", "dim_time", "dim_location", "dim_weather", "dim_road", "dim_vehicle", "dim_severity", "dim_congestion_level", "fact_accident", "fact_traffic"]:
        df = tables[name]
        cols = ", ".join(f"`{c}`" for c in df.columns)
        for start in range(0, len(df), 500):
            rows = [row for row in df.iloc[start:start+500].itertuples(index=False, name=None)]
            vals = ",\n".join("(" + ", ".join(sql_val(v) for v in row) + ")" for row in rows)
            f.write(f"INSERT INTO {name} ({cols}) VALUES\n{vals};\n")
    f.write("\nSET FOREIGN_KEY_CHECKS = 1;\n")
print("Saved to output/powerbi/ and sql/02_insert_data.sql")

## 6. Predictions

In [ ]:
df = acc.merge(dim_time[["hour", "time_band", "peak_period"]], on="hour")
df["dow_num"] = df.event_ts.dt.dayofweek
df["month"] = df.event_ts.dt.month
df = df.sort_values("event_ts").reset_index(drop=True)
overall = df.is_high_severity.mean()
print(f"Severe/Fatal rate: {100*overall:.1f}%")

rows = []
for col in ["location", "weather", "road_condition", "vehicle_type", "traffic_density", "time_band", "peak_period"]:
    p = chi2_contingency(pd.crosstab(df[col], df.is_high_severity))[1]
    rows.append({"factor": col, "p_value": round(p, 4), "linked": p < 0.05})
factors = pd.DataFrame(rows).sort_values("p_value")
factors.to_csv("output/powerbi/pred_factor_significance.csv", index=False)
factors

In [ ]:
risk = df.groupby(["location", "time_band"]).agg(n=("accident_id", "count"), severe=("is_high_severity", "sum")).reset_index()
risk["risk_index"] = (((risk.severe + 30*overall) / (risk.n + 30)) / overall).round(3)
risk = risk.sort_values("risk_index", ascending=False)
risk.to_csv("output/powerbi/pred_risk_location_timeband.csv", index=False)
risk.head(10)

In [ ]:
features = ["location", "weather", "road_condition", "traffic_density", "hour", "dow_num", "month"]
text_cols = ["location", "weather", "road_condition", "traffic_density"]
X = pd.get_dummies(df[features], columns=text_cols).astype(float)
y = df.is_high_severity
cut = int(len(df) * 0.8)
model = RandomForestClassifier(n_estimators=200, min_samples_leaf=25, class_weight="balanced", random_state=42)
model.fit(X[:cut], y[:cut])
auc = roc_auc_score(y[cut:], model.predict_proba(X[cut:])[:, 1])
print(f"ROC-AUC: {auc:.3f}")
pd.DataFrame({"model": ["Random Forest"], "roc_auc": [round(auc, 3)]}).to_csv("output/powerbi/pred_model_results.csv", index=False)
importance = pd.DataFrame({"feature": X.columns, "importance": model.feature_importances_.round(4)}).sort_values("importance", ascending=False)
importance.to_csv("output/powerbi/pred_feature_importance.csv", index=False)
importance.head(10)

In [ ]:
final = RandomForestClassifier(n_estimators=200, min_samples_leaf=25, class_weight="balanced", random_state=42).fit(X, y)
sample = df[features].sample(min(len(df), 500), random_state=0)
rows = []
for loc in LOCATIONS:
    for hr in range(24):
        t = sample.copy(); t["location"] = loc; t["hour"] = hr
        t = pd.get_dummies(t, columns=text_cols).reindex(columns=X.columns, fill_value=0).astype(float)
        rows.append({"location": loc, "hour": hr, "prob_severe": round(final.predict_proba(t)[:, 1].mean(), 4)})
forecast = pd.DataFrame(rows).sort_values("prob_severe", ascending=False).reset_index(drop=True)
forecast["rank"] = range(1, len(forecast) + 1)
forecast.to_csv("output/powerbi/pred_risk_forecast_location_hour.csv", index=False)
forecast.head(10)

In [ ]:
hotspots = sen.groupby(["location", "hour"]).agg(avg_congestion=("congestion_index", "mean"), avg_speed=("average_speed", "mean"),
                                                   readings=("reading_id", "count")).round(1).reset_index()
hotspots = hotspots.sort_values("avg_congestion", ascending=False)
hotspots.to_csv("output/powerbi/pred_congestion_hotspots.csv", index=False)
print(hotspots.head(10).to_string(index=False))

## Done
All CSVs saved to `output/powerbi/`. Run the MySQL scripts next.